# 09_Cross_Attention_Fusion_v2.ipynb
Robust cross-attention fusion with event_id normalization and merge validation.

In [ ]:
!pip -q install torch pandas scikit-learn

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
from google.colab import files

print("Upload cnn_features.csv")
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print("Upload transformer_features.csv")
u=files.upload()
tr=pd.read_csv(list(u.keys())[0])

def clean_event_id(s):
    return (s.astype(str)
             .str.replace("_Sentinel-2","",regex=False)
             .str.replace("_Landsat","",regex=False)
             .str.strip())

cnn["event_id"]=clean_event_id(cnn["event_id"])
tr["event_id"]=clean_event_id(tr["event_id"])

cnn=cnn.rename(columns={c:f"cnn_{c}" for c in cnn.columns if c!="event_id"})
tr=tr.rename(columns={c:f"tr_{c}" for c in tr.columns if c!="event_id"})

df=cnn.merge(tr,on="event_id",how="inner")

print("Matched Events:",len(df))

if len(df)==0:
    raise ValueError("No matching event_id values found after merge.")

labels=df["event_id"].apply(lambda x:"FLASH" if "FLASH" in x.upper() else ("HEAT" if "HEAT" in x.upper() else "OTHER"))
le=LabelEncoder()
y=le.fit_transform(labels)

cnn_cols=[c for c in df.columns if c.startswith("cnn_")]
tr_cols=[c for c in df.columns if c.startswith("tr_")]

class FusionDataset(Dataset):
    def __init__(self):
        self.cnn=torch.tensor(df[cnn_cols].values,dtype=torch.float32)
        self.tr=torch.tensor(df[tr_cols].values,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.cnn[i],self.tr[i],self.y[i]

dataset=FusionDataset()
train_size=max(1,int(0.8*len(dataset)))
val_size=len(dataset)-train_size
train_ds,val_ds=random_split(dataset,[train_size,val_size],generator=torch.Generator().manual_seed(42))
train_loader=DataLoader(train_ds,batch_size=8,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=8)

class FusionModel(nn.Module):
    def __init__(self,dim=512,ncls=2):
        super().__init__()
        self.attn=nn.MultiheadAttention(dim,8,batch_first=True)
        self.head=nn.Sequential(
            nn.Linear(dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,ncls)
        )
    def forward(self,cnn,tr):
        out,_=self.attn(cnn.unsqueeze(1),tr.unsqueeze(1),tr.unsqueeze(1))
        return self.head(out.squeeze(1))

device="cuda" if torch.cuda.is_available() else "cpu"
model=FusionModel(ncls=len(le.classes_)).to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-4)
loss_fn=nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    corr=tot=0
    for c,t,l in train_loader:
        c,t,l=c.to(device),t.to(device),l.to(device)
        opt.zero_grad()
        pred=model(c,t)
        loss=loss_fn(pred,l)
        loss.backward()
        opt.step()
        corr+=(pred.argmax(1)==l).sum().item()
        tot+=l.size(0)
    train_acc=100*corr/max(1,tot)

    model.eval()
    vc=vt=0
    with torch.no_grad():
        for c,t,l in val_loader:
            c,t,l=c.to(device),t.to(device),l.to(device)
            p=model(c,t)
            vc+=(p.argmax(1)==l).sum().item()
            vt+=l.size(0)
    val_acc=100*vc/max(1,vt)
    print(f"Epoch {epoch+1}: Train {train_acc:.2f}%  Val {val_acc:.2f}%")

torch.save(model.state_dict(),"multimodal_model.pth")
files.download("multimodal_model.pth")
print("Done.")
